In [1]:
# Parameters
BATCH_MODE = "true"

# Complexity-05c — La permanente, les circuits arithmétiques et l'annonce VP ≠ VNP

**Série Complexity — accrétion de 05** · Public : **Recherche** · Hommage : **Leslie Valiant** (les classes VP et VNP [Val79, Val82], et la complétude de la permanente sur tout corps où $2 \neq 0$ [Val79]) et **Herbert Ryser** (la formule de la permanente qui porte son nom, [Rys63]) · Source : le préprint *VP ≠ VNP over every field*, publié par Anthropic avec sa formalisation Lean ([anthropics/formal-math](https://github.com/anthropics/formal-math), projet `vp-vnp`, commit `11b6c7e3`, 23 novembre 2026)

**Navigation** : [Index de la série](README.md) | [05 — Compter est plus dur que vérifier](Complexity-05-CountingHarder-Permanent-Python.ipynb) | [05, version C#](Complexity-05-CountingHarder-Permanent-CSharp.ipynb) | [05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) | [05b, version C#](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-CSharp.ipynb)

> **Bandeau d'honnêteté.** Ce carnet sépare quatre choses qui se ressemblent et ne se valent pas : ce que nous **calculons ici** (des circuits arithmétiques réels et leurs comptes de portes), l'**énoncé formel** publié par les auteurs (lu au commit `11b6c7e3c927df2658f513b6b68ea7f8929c62c5`, cité verbatim au §10), leur **annonce** dans l'article (copie archivée au gisement, voir les références), et la **vérification indépendante** de la preuve — que nous ne faisons pas, et que les auteurs eux-mêmes déclarent non faite par un mathématicien humain (« No human mathematician has checked the proofs. », p. 11, et annexe Q, p. 66). Rien ici n'établit que P ≠ NP : l'énoncé séparé est algébrique, et ses auteurs écrivent eux-mêmes qu'il ne tranche pas la question booléenne (p. 1). Aucun `lake build` n'est lancé sur ce dépôt.

## 0. Pourquoi ce carnet

Le palier [05](Complexity-05-CountingHarder-Permanent-Python.ipynb) a posé le fait central : la **permanente** d'une matrice compte ses couplages parfaits pondérés, et ce comptage est #P-complet, là où le **déterminant** — la même somme, mais signée — se calcule en temps polynomial. [05b](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) en a tiré la frontière quantique : la même permanente, au cœur du `BosonSampling` d'Aaronson et Arkhipov (2011).

Ce carnet prolonge l'arc vers ce que la série n'avait pas encore : **la permanente vue comme polynôme, et les circuits arithmétiques qui la calculent**. Un circuit arithmétique est un programme droit qui n'a que deux opérations, `+` et `×`, et dont on compte les portes. On y admire un contraste exact : le déterminant a des circuits de taille polynomiale, la permanente n'en a pas de connue, et un préprint (dépôt `anthropics/formal-math`, lu au commit cité en tête de ce carnet) annonce une borne inférieure superpolynomiale. Nous construisons les circuits nous-mêmes, nous les évaluons, nous comparons leur taille à la borne annoncée — puis nous lisons l'énoncé formel, sans le vérifier.

**Prérequis ajoutés par rapport à 05** : permutations et signature ; arithmétique modulo un nombre premier ; la notion de polynôme à plusieurs indéterminées. Aucun prérequis de théorie de la complexité au-delà du vocabulaire de 05 (P, NP, #P) : les classes VP et VNP sont définies au §10, telles que le formulaire les définit.

**Durée indicative** : une heure et demie, exercices compris.

## 1. Quatre niveaux à ne pas confondre

Tout ce carnet parle d'un même objet — la permanente, et une borne inférieure annoncée pour elle — à quatre niveaux de confiance différents. Les mélanger est la faute que la série combat depuis [03d](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb).

| Niveau | Ce que c'est | Où le lecteur le vérifie |
|---|---|---|
| **Exécuté ici** | Des circuits construits par le code de ce carnet, évalués et comparés à une définition indépendante ; des tailles mesurées. | Les sorties des cellules, en particulier aux §5 à §9, rejouables à l'identique (graine fixe). |
| **Énoncé formel** | Ce que le dépôt `vp-vnp` affirme, écrit en Lean 4 : définitions et théorèmes. | Le texte verbatim au §10 ; le fichier au commit cité ; l'article, qui énonce les mêmes résultats à sa pagination (théorème 1.1 et corollaire 1.2, p. 2 ; définitions de VP et VNP, p. 4). |
| **Annonce des auteurs** | Ce que l'article revendique, ses théorèmes numérotés, ses limites déclarées. | La copie archivée au gisement (références), citée avec sa pagination : théorème 1.1 et corollaire 1.2 p. 2, identité de Ryser p. 60, corollaire M.4 p. 62, annexe Q p. 66 à 70. |
| **Vérification indépendante** | La preuve elle-même, et la question de savoir si les énoncés disent bien ce qu'ils prétendent dire. | **Hors de portée de ce carnet.** Les auteurs déclarent que la preuve n'a pas été contrôlée par un mathématicien humain (p. 11, et annexe Q). |

La règle de lecture qui en découle : une phrase de ce carnet qui cite un résultat des auteurs est une **citation**, jamais un acquis. Ce qui est acquis, c'est ce que les cellules impriment.

In [2]:
import math
import random as GENERATEUR
from itertools import permutations, product

# Toutes les experiences de ce carnet passent par un generateur a graine fixe :
# les matrices tirees ici se rejouent a l'identique sur n'importe quelle machine.
SEED = 20261124  # graine fixe : les tirages se rejouent a l'identique

print("Imports OK : math, random, itertools")
print(f"SEED = {SEED} -- les instances sont rejouables a l'identique.")

Imports OK : math, random, itertools
SEED = 20261124 -- les instances sont rejouables a l'identique.


## 2. La permanente et le déterminant, par définition

Pour une matrice $A = (a_{ij})$ de taille $m \times m$ :

$$\mathrm{perm}(A) = \sum_{\sigma \in S_m} \prod_{i=1}^{m} a_{i,\sigma(i)} \qquad \det(A) = \sum_{\sigma \in S_m} \mathrm{sgn}(\sigma) \prod_{i=1}^{m} a_{i,\sigma(i)}$$

Deux sommes sur les mêmes $m!$ permutations ; la seconde seule porte un signe. Toute la différence de difficulté tient dans ce signe, et le §8 montrera qu'il disparaît sur un corps de caractéristique 2.

Travailler « sur un corps » demande de décider lequel. Pour que les mêmes fonctions tournent sur $\mathbb{Z}$ et sur $\mathbb{F}_p$, on encapsule l'arithmétique dans un petit objet `Anneau` : réduire modulo $p$, ou ne rien réduire du tout.

In [3]:
class Anneau:
    """Z (p = 0) ou Z/pZ, p premier : les memes fonctions tournent sur les deux."""

    def __init__(self, p: int = 0) -> None:
        self.p = p

    def __repr__(self) -> str:
        return "Z" if self.p == 0 else "F_%d" % self.p

    def reduire(self, a: int) -> int:
        return a % self.p if self.p else a

    def zero(self) -> int:
        return 0

    def un(self) -> int:
        return self.reduire(1)

    def add(self, a: int, b: int) -> int:
        return self.reduire(a + b)

    def mul(self, a: int, b: int) -> int:
        return self.reduire(a * b)

    def oppose(self, a: int) -> int:
        return self.reduire(-a)

    def egal(self, a: int, b: int) -> bool:
        return self.reduire(a - b) == 0


def signature(sigma) -> int:
    """Signature d'une permutation : +1 ou -1, par comptage des cycles."""
    m = len(sigma)
    vu = [False] * m
    cycles = 0
    for i in range(m):
        if not vu[i]:
            cycles += 1
            j = i
            while not vu[j]:
                vu[j] = True
                j = sigma[j]
    return -1 if (m - cycles) % 2 else 1


def permanente_par_definition(A, anneau: Anneau):
    """perm(A) = somme sur les permutations sigma de prod_i A[i][sigma(i)]."""
    total = anneau.zero()
    for sigma in permutations(range(len(A))):
        terme = anneau.un()
        for i in range(len(A)):
            terme = anneau.mul(terme, A[i][sigma[i]])
        total = anneau.add(total, terme)
    return total


def determinant_par_definition(A, anneau: Anneau):
    """det(A) = somme sur les permutations sigma de signature(sigma) * prod_i A[i][sigma(i)]."""
    total = anneau.zero()
    for sigma in permutations(range(len(A))):
        terme = anneau.un()
        for i in range(len(A)):
            terme = anneau.mul(terme, A[i][sigma[i]])
        if signature(sigma) < 0:
            terme = anneau.oppose(terme)
        total = anneau.add(total, terme)
    return total


def matrice_aleatoire(m: int, anneau: Anneau, generateur, amplitude: int = 6) -> list:
    return [[anneau.reduire(generateur.randrange(-amplitude, amplitude + 1))
             for _ in range(m)] for _ in range(m)]


A = [[1, 2], [3, 4]]
print("matrice A =", A)
for anneau in (Anneau(0), Anneau(2), Anneau(3), Anneau(5)):
    print(f"   sur {str(anneau):4s} : perm = {permanente_par_definition(A, anneau):3d}"
          f"   det = {determinant_par_definition(A, anneau):3d}")
print()
B = [[0, 1, 1], [1, 0, 1], [1, 1, 0]]
print("matrice B (matrice d'adjacence du triangle K3) =", B)
print(f"   sur Z    : perm = {permanente_par_definition(B, Anneau(0))}"
      f"   det = {determinant_par_definition(B, Anneau(0))}")
print("   (perm(B) = 2 : les deux derangements de trois elements, soit les deux orientations,")
print("    c'est-a-dire les deux cycles hamiltoniens orientes de K3 ; lue comme matrice de")
print("    biadjacence d'un graphe biparti, B y compte aussi deux couplages parfaits)")

matrice A = [[1, 2], [3, 4]]
   sur Z    : perm =  10   det =  -2
   sur F_2  : perm =   0   det =   0
   sur F_3  : perm =   1   det =   1
   sur F_5  : perm =   0   det =   3

matrice B (matrice d'adjacence du triangle K3) = [[0, 1, 1], [1, 0, 1], [1, 1, 0]]
   sur Z    : perm = 2   det = 2
   (perm(B) = 2 : les deux derangements de trois elements, soit les deux orientations,
    c'est-a-dire les deux cycles hamiltoniens orientes de K3 ; lue comme matrice de
    biadjacence d'un graphe biparti, B y compte aussi deux couplages parfaits)


### Lecture du résultat : deux sommes, deux nombres, quatre arithmétiques

Pour $A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}$, la permanente vaut 10 et le déterminant $-2$ sur $\mathbb{Z}$. Les deux sommes parcourent les **mêmes** deux permutations : l'identité apporte $1 \times 4$, l'échange apporte $2 \times 3$. Seule change la signature — l'échange est impair, et son terme change de signe : $4 + 6 = 10$ d'un côté, $4 - 6 = -2$ de l'autre.

Le même calcul sur les trois corps finis montre que le résultat dépend de l'arithmétique et pas seulement de la matrice. Sur $\mathbb{F}_2$, les deux nombres tombent à 0 ; sur $\mathbb{F}_3$, les deux valent 1 — une coïncidence d'arithmétique, $10 \equiv -2 \equiv 1 \pmod 3$, et non une identité de polynômes ; sur $\mathbb{F}_5$, ils se séparent à nouveau, 0 contre 3. Le §8 cherchera systématiquement pour quelles caractéristiques les deux **polynômes** coïncident : il n'en trouvera qu'une, la caractéristique 2.

La matrice $B$, elle, n'a que des 0 et des 1 : c'est la matrice d'adjacence du triangle. Ses deux nombres sont **égaux** sur $\mathbb{Z}$, et la raison est structurelle — seules deux permutations apportent un terme non nul, les deux cycles de longueur trois, tous deux de signature $+1$. Lue comme matrice de biadjacence d'un graphe biparti, $B$ compte aussi deux couplages parfaits : la permanente d'une matrice 0/1 est un **compte**, et c'est le pont que l'exercice 1 demandera de construire.

## 3. Le polynôme du cycle hamiltonien

La permanente est une **fonction** ; pour parler de circuits arithmétiques, il faut la voir comme un **polynôme** en les $m^2$ indéterminées $X_{u,v}$. Trois polynômes vont nous servir :

- la permanente générique $\sum_\sigma \prod_i X_{\sigma(i),i}$ ;
- le déterminant générique, le même avec $\mathrm{sgn}(\sigma)$ ;
- et $\mathrm{hc}_m$ — le **polynôme du cycle hamiltonien** — qui ne somme que les permutations constituées d'**un seul** cycle de longueur $m$.

Une permutation d'un seul cycle de longueur $m$ est un cycle hamiltonien sur le graphe complet orienté : $\mathrm{hc}_m$ compte les cycles hamiltoniens du graphe dont la matrice d'adjacence porte les indéterminées. Le §10 de ce carnet en donnera la définition exacte, recopiée du formulaire Lean ; ici, nous calculons.

Un monôme sans exposant est un **ensemble** de variables : on le code comme un `frozenset` d'entiers, la variable $X_{u,v}$ portant le numéro $u \cdot m + v$. Deux orientations sont possibles — $X_{\sigma(i),i}$, celle que le formulaire Lean écrit pour `perm`, et $X_{i,\sigma(i)}$, celle qu'il écrit pour `hc` — et l'annexe Q de l'article (p. 68) note que les deux sommes coïncident, en remplaçant $\sigma$ par son inverse. Nous vérifions ce point plutôt que de le croire.

In [4]:
def nombre_de_cycles(sigma) -> int:
    vu = [False] * len(sigma)
    cycles = 0
    for i in range(len(sigma)):
        if not vu[i]:
            cycles += 1
            j = i
            while not vu[j]:
                vu[j] = True
                j = sigma[j]
    return cycles


def monome(sigma, m: int, inverse: bool = False) -> frozenset:
    """Produit des X(sigma(i), i), ou des X(i, sigma(i)) si `inverse`."""
    return frozenset((sigma[i] * m + i) if not inverse else (i * m + sigma[i])
                     for i in range(m))


def polynome_perm(m: int, inverse: bool = False, signe: bool = False) -> dict:
    """La permanente generique (signe = False) ou le determinant generique (signe = True)."""
    p = {}
    for sigma in permutations(range(m)):
        mono = monome(sigma, m, inverse)
        p[mono] = p.get(mono, 0) + (signature(sigma) if signe else 1)
    return {k: v for k, v in p.items() if v}


def polynome_hc(m: int) -> dict:
    """hc_m : somme sur les permutations qui sont UN cycle de longueur m de leur monome."""
    p = {}
    for sigma in permutations(range(m)):
        if nombre_de_cycles(sigma) == 1:
            mono = monome(sigma, m)
            p[mono] = p.get(mono, 0) + 1
    return p


def reduire_modulo(p: dict, mod: int) -> dict:
    return {k: v % mod for k, v in p.items() if v % mod}


print("m  monomes de perm(A)   monomes de hc_m   (m-1)!")
for m in range(1, 9):
    print(f"{m:2d} {len(polynome_perm(m)):19d} {len(polynome_hc(m)):17d} {math.factorial(m - 1):7d}")
print()
print("orientation perm (X(sigma i, i) contre X(i, sigma i)) :",
      all(polynome_perm(m) == polynome_perm(m, inverse=True) for m in range(1, 7)))
print("orientation hc   :",
      all(polynome_hc(m) == {monome(s, m, True): 1 for s in permutations(range(m))
                             if nombre_de_cycles(s) == 1} for m in range(1, 7)))
print()
print("hc_3, cycle par cycle (monome -> coefficient) :")
for mono, coeff in sorted(polynome_hc(3).items()):
    print("   ", sorted(mono), "->", coeff)

m  monomes de perm(A)   monomes de hc_m   (m-1)!
 1                   1                 1       1
 2                   2                 1       1
 3                   6                 2       2
 4                  24                 6       6
 5                 120                24      24
 6                 720               120     120
 7                5040               720     720


 8               40320              5040    5040

orientation perm (X(sigma i, i) contre X(i, sigma i)) : True
orientation hc   : True

hc_3, cycle par cycle (monome -> coefficient) :
    [2, 3, 7] -> 1
    [1, 5, 6] -> 1


### Lecture du résultat : $m!$ monômes, $(m-1)!$ cycles

La première colonne compte les monômes de la permanente générique : 1, 2, 6, 24, 120, 720, 5040, puis 40320 à $m = 8$. C'est $m!$, un monôme par permutation, et ce nombre ne peut pas baisser : deux permutations distinctes diffèrent sur au moins un indice, donc font intervenir au moins une variable différente, et aucun monôme ne peut en recouvrir un autre. Écrire le polynôme sous forme développée coûte donc $m!$ monômes, là où le §5 construira un circuit de l'ordre de $2^m$ portes, à un facteur polynomial près.

Les deux colonnes suivantes coïncident ligne par ligne, et c'est ce qui est **mesuré** : les permutations de longueur $m$ qui forment un seul cycle, énumérées une par une, sont exactement au nombre de $(m-1)!$ — 1, 1, 2, 6, 24, 120, 720, 5040. Le polynôme $\mathrm{hc}_m$ a donc bien $(m-1)!$ monômes, un par cycle hamiltonien du graphe complet orienté.

Les deux lignes `orientation` valent `True`. Écrire $\prod_i X_{\sigma(i),\, i}$ — l'écriture de `perm` dans le formulaire Lean — ou $\prod_i X_{i,\, \sigma(i)}$ — celle de `hc` — définit **le même** polynôme : remplacer $\sigma$ par son inverse échange les deux écritures, et l'ensemble des permutations est stable par inversion. La vérification porte sur $m$ de 1 à 6, l'étendue que se donne la cellule. C'est l'écart d'écriture que l'annexe Q de l'article recense entre sa section 1 et son formulaire (§11).

Enfin `hc_3` montre les deux monômes du polynôme d'ordre 3, chacun de coefficient 1. Avec les variables numérotées $u \cdot m + v$, les indices 2, 3, 7 d'un côté et 1, 5, 6 de l'autre sont les deux cycles de longueur trois du triangle : l'un et son inverse.

## 4. Circuits arithmétiques : le modèle du papier

Un **circuit arithmétique** est une liste finie de portes. Chaque porte est soit une **variable** $X_i$, soit une **constante**, soit une **addition** de deux portes antérieures, soit une **multiplication** de deux portes antérieures. La sortie est la dernière porte. La **taille** est la longueur de la liste.

Trois détails du modèle comptent pour tous les comptes de ce carnet, et le formulaire Lean les fixe aussi :

1. **Les variables et les constantes comptent comme des portes.** Ce n'est pas une convention universelle — l'article note lui-même qu'un comptage limité aux additions et multiplications ne changerait la taille que d'un facteur au plus trois (note 2, p. 4, pour la source ; §9 du présent carnet pour la mesure de ce facteur).
2. **Il n'y a ni soustraction ni négation.** Un signe moins se paie : c'est un produit par la constante $-1$.
3. **Un circuit calcule un polynôme**, pas seulement une valeur : on l'évalue dans un anneau en donnant une valeur à chaque variable.

Le formulaire Lean du §10 fixe le même modèle, avec une nuance de lecture : sa définition `Computes` demande l'existence d'une liste de $s$ portes dont **l'une** est $f$ — pas nécessairement la dernière. C'est une convention plus permissive que la nôtre, et sans conséquence sur la taille : une liste peut toujours être tronquée juste après la porte qui porte $f$, donc la taille minimale est la même.

Nous écrivons deux « arithmétiques » de même interface : l'une manipule des **valeurs** d'un anneau, l'autre des **indices de portes** et construit le circuit. Le même code de construction tourne alors dans les deux modes, ce qui permet de comparer une construction à elle-même.

In [5]:
class Circuit:
    """Un circuit arithmetique : une liste de portes.

    Chaque porte est ('var', i), ('const', a), ('add', u, v) ou ('mul', u, v), ou u et v
    sont des indices de portes ANTERIEURES. La taille est la longueur de la liste :
    variables et constantes comptent comme des portes.
    """

    def __init__(self) -> None:
        self.portes: list = []

    def __len__(self) -> int:
        return len(self.portes)

    def _porte(self, p) -> int:
        self.portes.append(p)
        return len(self.portes) - 1

    def var(self, i: int) -> int:
        return self._porte(("var", i))

    def const(self, a) -> int:
        return self._porte(("const", a))

    def add(self, u: int, v: int) -> int:
        return self._porte(("add", u, v))

    def mul(self, u: int, v: int) -> int:
        return self._porte(("mul", u, v))

    def evaluer(self, entrees, anneau: Anneau, sortie=None):
        """Valeur de la porte `sortie` (par defaut la derniere) sur les `entrees`."""
        vals = []
        for p in self.portes:
            kind = p[0]
            if kind == "var":
                vals.append(entrees[p[1]])
            elif kind == "const":
                vals.append(anneau.reduire(p[1]))
            elif kind == "add":
                vals.append(anneau.add(vals[p[1]], vals[p[2]]))
            else:
                vals.append(anneau.mul(vals[p[1]], vals[p[2]]))
        return vals[sortie if sortie is not None else -1]

    def decompte(self) -> dict:
        """Nombre de portes par nature."""
        d = {"var": 0, "const": 0, "add": 0, "mul": 0}
        for p in self.portes:
            d[p[0]] += 1
        return d


class OperationsValeurs:
    """Arithmetique sur des valeurs d'un anneau."""

    def __init__(self, anneau: Anneau) -> None:
        self.anneau = anneau

    def constante(self, v: int) -> int:
        return self.anneau.reduire(v)

    def add(self, a: int, b: int) -> int:
        return self.anneau.add(a, b)

    def mul(self, a: int, b: int) -> int:
        return self.anneau.mul(a, b)

    def oppose(self, a: int) -> int:
        return self.anneau.oppose(a)


class OperationsCircuit:
    """La meme interface qu'OperationsValeurs : chaque operation ajoute une porte.

    Le modele n'a que l'addition et la multiplication. Un oppose est donc un produit
    par la constante -1, et une difference coute une porte de plus qu'un produit.
    """

    def __init__(self) -> None:
        self.circuit = Circuit()
        self._un = None
        self._moins_un = None

    @property
    def un(self) -> int:
        if self._un is None:
            self._un = self.circuit.const(1)
        return self._un

    @property
    def moins_un(self) -> int:
        if self._moins_un is None:
            self._moins_un = self.circuit.const(-1)
        return self._moins_un

    def constante(self, v: int) -> int:
        if v == 1:
            return self.un
        if v == -1:
            return self.moins_un
        return self.circuit.const(v)

    def entree(self, etiquette: int) -> int:
        return self.circuit.var(etiquette)

    def add(self, a: int, b: int) -> int:
        return self.circuit.add(a, b)

    def mul(self, a: int, b: int) -> int:
        return self.circuit.mul(a, b)

    def oppose(self, a: int) -> int:
        return self.circuit.mul(a, self.moins_un)


# (x0 + x1) * x2, construit porte par porte
ops = OperationsCircuit()
c = ops.circuit
p0 = ops.entree(0)
p1 = ops.entree(1)
p2 = ops.entree(2)
somme = c.add(p0, p1)
produit = c.mul(somme, p2)
print("apres (x0 + x1) * x2 :", len(c), "portes", c.decompte())
print("   en (2, 3, 4) sur Z   :", c.evaluer([2, 3, 4], Anneau(0)))
print("   en (1, 1, 1) sur F_2 :", c.evaluer([1, 1, 1], Anneau(2)))
negation = c.mul(p0, ops.moins_un)
print("apres avoir ajoute -x0 :", len(c), "portes", c.decompte())
print("   -x0 pour x0 = 5 sur Z :", c.evaluer([5, 0, 0], Anneau(0), sortie=negation))
print("   (la constante -1 s'ajoute une seule fois, puis elle est partagee)")

apres (x0 + x1) * x2 : 5 portes {'var': 3, 'const': 0, 'add': 1, 'mul': 1}
   en (2, 3, 4) sur Z   : 20
   en (1, 1, 1) sur F_2 : 0
apres avoir ajoute -x0 : 7 portes {'var': 3, 'const': 1, 'add': 1, 'mul': 2}
   -x0 pour x0 = 5 sur Z : -5
   (la constante -1 s'ajoute une seule fois, puis elle est partagee)


### Lecture du résultat : les entrées se paient

L'expression $(x_0 + x_1) \times x_2$ s'écrit avec deux opérations, mais le circuit en compte **cinq** portes, dont trois variables : chaque variable lue coûte une porte, chaque constante aussi, et chaque opération par-dessus. C'est le modèle que le §4 pose et que `Computes` écrira formellement au §10 — il ne compte pas les opérations seules, il compte tout ce qu'il faut écrire pour les produire.

Évalué en $(2, 3, 4)$ sur $\mathbb{Z}$, le circuit rend 20 ; évalué en $(1, 1, 1)$ sur $\mathbb{F}_2$, il rend 0. Le circuit n'a pas changé — mêmes portes, même ordre, même parcours — et pourtant le résultat change : $1 + 1$ vaut $0$ sur $\mathbb{F}_2$ et $2$ sur $\mathbb{Z}$. Le circuit est un objet **neutre**, c'est l'anneau qui décide. C'est pourquoi le §5 évalue le même circuit sur quatre arithmétiques de suite, sans le réécrire une seule fois.

La seconde moitié de la sortie montre le prix d'un signe moins. Ajouter $-x_0$ coûte exactement deux portes — une constante $-1$, puis une multiplication — et fait passer les multiplications de 1 à 2 ; le résultat vaut $-5$ pour $x_0 = 5$. La dernière ligne énonce la règle de partage : la constante $-1$ s'ajoute **une seule fois**, puis elle est réutilisée. Une deuxième négation ne coûterait donc qu'une porte de plus, pas deux — et c'est ce partage que le circuit de Ryser met à profit au §5 : une seule constante $-1$ pour toutes ses négations.

## 5. Ryser : un circuit réel pour la permanente

La définition par les $m!$ permutations est inutilisable comme circuit : sa taille croît comme $m!$. L'identité de **Ryser** ramène la permanente à une somme sur les $2^m$ **sous-ensembles** de colonnes :

$$\mathrm{perm}(A) = (-1)^m \sum_{\varnothing \neq S \subseteq [m]} (-1)^{|S|} \prod_{i=1}^{m} \Big( \sum_{j \in S} a_{ij} \Big)$$

Elle n'est pas une simplification miraculeuse : $2^m$ est exponentiel, et l'article rappelle (p. 1, références `[Rys63, Gly10, Cur26]`) que les meilleures bornes **supérieures** connues sont de cet ordre, de la forme $2^{O(m)}$ ; nous rapportons cette phrase sans avoir relu ces trois travaux. Mais c'est une construction **réelle**, dont on peut compter les portes exactement.

Deux optimisations rendent le compte lisible : les sommes $T_i(S)$ sont **partagées** entre sous-ensembles emboîtés ($T_i(S) = T_i(S \setminus \{j\}) + a_{ij}$ pour $j$ le plus petit élément de $S$), et la constante $-1$ n'est créée qu'une fois. Chaque négation coûte alors une multiplication.

La cellule vérifie deux choses : que la taille construite égale une **forme close** calculée à part, et que le circuit **évalue** bien la permanente de la définition — sur quatre anneaux.

In [6]:
def circuit_ryser(m: int) -> Circuit:
    """Circuit calculant la permanente d'une matrice m x m, par la formule de Ryser.

    perm(A) = (-1)^m * somme_{S inclus dans [m], S non vide} (-1)^{|S|} prod_i somme_{j dans S} a_ij
    Entrees : les m^2 coefficients, lus ligne par ligne (a_ij a l'indice i * m + j).
    T_i(S) = T_i(S \\ {j}) + a_ij pour j le plus petit element de S : les sommes par
    sous-ensemble sont partagees, une addition par couple (ligne, sous-ensemble).
    """
    ops = OperationsCircuit()
    c = ops.circuit
    x = [[ops.entree(i * m + j) for j in range(m)] for i in range(m)]
    sommes = {}
    termes = []
    for masque in range(1, 1 << m):
        bit = masque & -masque
        j = bit.bit_length() - 1
        reste = masque ^ bit
        taille = bin(masque).count("1")
        par_ligne = []
        for i in range(m):
            if reste == 0:
                par_ligne.append(x[i][j])
            else:
                par_ligne.append(c.add(sommes[(reste, i)], x[i][j]))
            sommes[(masque, i)] = par_ligne[i]
        produit = par_ligne[0]
        for i in range(1, m):
            produit = c.mul(produit, par_ligne[i])
        if (taille + m) % 2:            # signe (-1)^(m + |S|) : negation par produit par -1
            produit = c.mul(produit, ops.moins_un)
        termes.append(produit)
    total = termes[0]
    for t in termes[1:]:
        total = c.add(total, t)
    return c


def taille_ryser(m: int) -> int:
    """Nombre de portes du circuit construit par circuit_ryser, en forme close."""
    return (m * m                              # variables
            + (1 if m >= 2 else 0)             # constante -1 (inutile pour m = 1)
            + m * ((1 << m) - 1 - m)           # additions des sommes par sous-ensemble
            + (m - 1) * ((1 << m) - 1)         # multiplications des produits
            + ((1 << (m - 1)) - (m % 2))       # negations
            + ((1 << m) - 2))                  # additions de la somme finale


def taille_ryser_en_une_ligne(m: int) -> int:
    """La meme taille, reduite : (4m + 1) 2^(m - 1) - 2m - 1 + [m >= 2] - (m mod 2)."""
    return (4 * m + 1) * (1 << (m - 1)) - 2 * m - 1 + (1 if m >= 2 else 0) - m % 2


gen = GENERATEUR.Random(SEED)
anneaux = [Anneau(0), Anneau(2), Anneau(3), Anneau(5)]
echecs = 0
for anneau in anneaux:
    for m in range(1, 6):
        circuit = circuit_ryser(m)
        for _ in range(10):
            A = matrice_aleatoire(m, anneau, gen, 4)
            plat = [A[i][j] for i in range(m) for j in range(m)]
            if not anneau.egal(circuit.evaluer(plat, anneau), permanente_par_definition(A, anneau)):
                echecs += 1
print("circuit de Ryser contre la definition :", len(anneaux) * 5 * 10, "essais,", echecs, "echec(s)")
ecarts = [m for m in range(1, 13) if len(circuit_ryser(m)) != taille_ryser(m)]
print("taille construite egale a la forme close pour m = 1..12 :", not ecarts, "(ecarts :", ecarts, ")")
ecarts_ligne = [m for m in range(1, 13) if len(circuit_ryser(m)) != taille_ryser_en_une_ligne(m)]
print("forme close reduite a une ligne, egale aussi pour m = 1..12 :", not ecarts_ligne,
      "(ecarts :", ecarts_ligne, ")")
print()
print("m   portes   log2(portes)")
for m in (1, 2, 3, 4, 5, 6, 8, 10, 12):
    print(f"{m:2d} {taille_ryser(m):8d} {math.log2(taille_ryser(m)):12.3f}")
print()
print("decompte a m = 6 :", circuit_ryser(6).decompte())

circuit de Ryser contre la definition : 200 essais, 0 echec(s)


taille construite egale a la forme close pour m = 1..12 : True (ecarts : [] )
forme close reduite a une ligne, egale aussi pour m = 1..12 : True (ecarts : [] )

m   portes   log2(portes)
 1        1        0.000
 2       14        3.807
 3       45        5.492
 4      128        7.000
 5      325        8.344
 6      788        9.622
 8     4208       12.039
10    20972       14.356
12   100328       16.614



decompte a m = 6 :

 {'var': 36, 'const': 1, 'add': 404, 'mul': 347}


### Lecture du résultat : 200 essais, deux formes closes, une croissance en $2^m$

Les 200 essais, c'est quatre anneaux fois cinq tailles fois dix matrices tirées. À chaque fois, le circuit de Ryser est évalué et comparé à la permanente calculée par la définition, et aucun des 200 ne diverge. Les deux lignes suivantes ne sont pas des approximations : `True` avec une liste d'écarts **vide** signifie que la taille construite et la première forme close coïncident exactement, pour $m = 1$ à $12$, et que la forme close réduite à une ligne coïncide, elle aussi, avec la taille construite. Une forme close fausse d'une seule porte aurait rempli cette liste.

La colonne $\log_2$ montre la croissance : 7.000 à $m = 4$, puis 16.614 à $m = 12$ — un peu plus d'une unité par pas de $m$. La taille est en effet, pour l'essentiel, $2^m$ fois un terme de l'ordre de $m$ : la somme porte sur les $2^m$ sous-ensembles de colonnes, et chacun coûte un produit de $m$ termes.

Le décompte à $m = 6$ détaille ce total : 36 variables, une seule constante, 404 additions et 347 multiplications. Les additions sont les plus nombreuses parce qu'elles sont, pour l'essentiel, les sommes partielles $T_i(S)$ — une par couple (ligne, sous-ensemble d'au moins deux colonnes) —, le reste étant la somme finale des termes. Et la constante unique est le $-1$ des négations, créé une fois puis réutilisé.

## 6. Le déterminant, lui, a des circuits polynomiaux

Le contraste est là. Pour le déterminant, on peut écrire une construction **sans aucune division**, donc valable sur un anneau commutatif quelconque, en récursant sur les coefficients du polynôme caractéristique. Si $B$ est la sous-matrice principale d'ordre $k-1$, $b$ la colonne $k$ au-dessus de la diagonale et $a$ la ligne $k$ à sa gauche, et si $\mu_i = a^{\top} B^{i-1} b$, alors

$$c^{(k)}_{\ell} = c^{(k-1)}_{\ell} - a_{kk}\, c^{(k-1)}_{\ell-1} - \sum_{i \ge 1} \mu_i\, c^{(k-1)}_{\ell-i-1}$$

et $\det A = (-1)^m c^{(m)}_{m}$. Chaque $\mu_i$ demande un produit matrice-vecteur : le coût est polynomial en $m$ — **de degré 4** pour cette construction, comme la cellule suivante le mesure — et c'est le même code qui, branché sur `OperationsCircuit`, produit le circuit.

L'article, lui, attribue au déterminant des circuits de taille polynomiale (`[Ber84]`, p. 2, §1.2 ; repris p. 10, §2.5), et rappelle que la meilleure borne **inférieure** antérieure pour la permanente était de l'ordre de $\Omega(n \log d)$ (`[Str73, BS83]`, p. 2). Ces deux attributions sont celles de l'article, rapportées sans relecture des sources : notre récursion est une construction de plus, pas une attribution nouvelle — nous ne prétendons pas reproduire l'identité publiée de quiconque.

In [7]:
def produit_scalaire(u, v, ops):
    """Somme des produits terme a terme : len(u) - 1 additions et len(u) multiplications."""
    acc = ops.mul(u[0], v[0])
    for a, b in zip(u[1:], v[1:]):
        acc = ops.add(acc, ops.mul(a, b))
    return acc


def determinant_par_recursion(M, ops):
    """Determinant par recursion sur les coefficients du polynome caracteristique, sans division.

    Avec ops = OperationsValeurs(anneau) : une valeur. Avec ops = OperationsCircuit() :
    l'indice de la porte de sortie du circuit, `M` etant alors la matrice des indices
    des variables.
    """
    m = len(M)
    if m == 0:
        return ops.constante(1)
    c = [ops.constante(1), ops.oppose(M[0][0])]
    for k in range(2, m + 1):
        n = k - 1
        b = [M[j][k - 1] for j in range(n)]
        a = [M[k - 1][j] for j in range(n)]
        B = [ligne[:n] for ligne in M[:n]]
        v = list(b)
        mu = [produit_scalaire(a, v, ops)]
        for _ in range(2, n + 1):
            v = [produit_scalaire(B[r], v, ops) for r in range(n)]
            mu.append(produit_scalaire(a, v, ops))
        oppose_akk = ops.oppose(M[k - 1][k - 1])
        opposes_mu = [ops.oppose(x) for x in mu]
        nc = [ops.constante(1)]
        for l in range(1, k + 1):
            termes = []
            if l <= n:
                termes.append(c[l])
            termes.append(ops.mul(oppose_akk, c[l - 1]))
            for i in range(1, min(n, l - 1) + 1):
                termes.append(ops.mul(opposes_mu[i - 1], c[l - i - 1]))
            acc = termes[0]
            for t in termes[1:]:
                acc = ops.add(acc, t)
            nc.append(acc)
        c = nc
    return c[m] if m % 2 == 0 else ops.oppose(c[m])


def circuit_determinant(m: int) -> Circuit:
    """Circuit calculant le determinant d'une matrice m x m (entrees lues ligne par ligne)."""
    ops = OperationsCircuit()
    determinant_par_recursion([[ops.entree(i * m + j) for j in range(m)] for i in range(m)], ops)
    return ops.circuit


def taille_determinant(m: int) -> int:
    """Nombre de portes du circuit construit par circuit_determinant, en forme close."""
    total = m * m + 2 + 1 + (m % 2)
    for n in range(1, m):
        total += (n ** 3 + n * (n - 1) // 2 + 3 * n + 2)      # multiplications
        total += n * n * (n - 1) + n * (n - 1) // 2 + 2 * n    # additions
    return total


def taille_determinant_polynomiale(m: int) -> int:
    """La meme taille, ecrite comme polynome en m : m^2 (m^2 + 7) / 2 - m^3 + 1 + (m mod 2)."""
    return m * m * (m * m + 7) // 2 - m ** 3 + 1 + m % 2


gen = GENERATEUR.Random(SEED + 1)
anneaux = [Anneau(0), Anneau(2), Anneau(3), Anneau(5)]
echecs_valeurs = 0
echecs_circuit = 0
for anneau in anneaux:
    for m in range(1, 7):
        for _ in range(10):
            A = matrice_aleatoire(m, anneau, gen, 4)
            if not anneau.egal(determinant_par_recursion(A, OperationsValeurs(anneau)),
                               determinant_par_definition(A, anneau)):
                echecs_valeurs += 1
            circuit = circuit_determinant(m)
            plat = [A[i][j] for i in range(m) for j in range(m)]
            if not anneau.egal(circuit.evaluer(plat, anneau), determinant_par_definition(A, anneau)):
                echecs_circuit += 1
print("recursion sur les valeurs contre la definition :", len(anneaux) * 6 * 10, "essais,",
      echecs_valeurs, "echec(s)")
print("circuit evalue contre la definition          :", len(anneaux) * 6 * 10, "essais,",
      echecs_circuit, "echec(s)")
ecarts = [m for m in range(1, 13) if len(circuit_determinant(m)) != taille_determinant(m)]
print("taille construite egale a la forme close pour m = 1..12 :", not ecarts, "(ecarts :", ecarts, ")")
ecarts_poly = [m for m in range(1, 201)
               if taille_determinant(m) != taille_determinant_polynomiale(m)]
print("forme close polynomiale egale a la forme en boucle pour m = 1..200 :",
      not ecarts_poly, "(ecarts :", ecarts_poly, ")")
print()
print("m   determinant   Ryser    rapport")
for m in (1, 2, 3, 4, 5, 6, 8, 10, 12):
    print(f"{m:2d} {taille_determinant(m):13d} {taille_ryser(m):8d} {taille_ryser(m) / taille_determinant(m):9.1f}")
print()
print("doublement de m : facteur sur la taille du circuit")
for petit, grand in ((3, 6), (4, 8), (6, 12)):
    print(f"   m {petit:2d} -> {grand:2d} : determinant x"
          f"{taille_determinant(grand) / taille_determinant(petit):6.1f}"
          f"   Ryser x{taille_ryser(grand) / taille_ryser(petit):7.1f}")

recursion sur les valeurs contre la definition : 240 essais, 0 echec(s)
circuit evalue contre la definition          : 240 essais, 0 echec(s)
taille construite egale a la forme close pour m = 1..12 : True (ecarts : [] )
forme close polynomiale egale a la forme en boucle pour m = 1..200 : True (ecarts : [] )

m   determinant   Ryser    rapport
 1             5        1       0.2
 2            15       14       0.9
 3            47       45       1.0
 4           121      128       1.1
 5           277      325       1.2
 6           559      788       1.4
 8          1761     4208       2.4
10          4351    20972       4.8
12          9145   100328      11.0

doublement de m : facteur sur la taille du circuit
   m  3 ->  6 : determinant x  11.9   Ryser x   17.5
   m  4 ->  8 : determinant x  14.6   Ryser x   32.9
   m  6 -> 12 : determinant x  16.4   Ryser x  127.3


### Lecture du résultat : les deux courbes se croisent à $m = 4$

Les deux premières lignes testent la même récursion dans ses deux modes : 240 essais sur des **valeurs**, puis 240 essais à travers le **circuit** que le même code construit, et aucun échec dans l'un ni dans l'autre. Elles ne sont pas redondantes : la première valide la récursion sur des valeurs, la seconde valide le circuit — si le mode circuit déformait le calcul, la première resterait verte et la seconde non.

Vient ensuite le contraste que la section annonce. Aux tailles 1, 2 et 3, le déterminant coûte **plus** de portes que Ryser — 5 contre 1, 15 contre 14, 47 contre 45 : sa construction a des frais fixes, les constantes et les produits par $-1$, qu'une matrice trop petite n'amortit pas. Le croisement a lieu à $m = 4$ (128 portes contre 121) et l'écart ne cesse ensuite de grandir : à $m = 12$, Ryser vaut 11.0 fois la taille du déterminant, 100328 portes contre 9145.

Le dernier bloc explique ce croisement mieux qu'un rapport ponctuel. Quand $m$ double, le circuit du déterminant est multiplié par un facteur qui grandit lentement — 11.9, puis 14.6, puis 16.4 — la signature d'un polynôme de degré 4, dont le facteur de doublement tend vers 16. Le circuit de Ryser, lui, est multiplié par 17.5, puis 32.9, puis 127.3 : à chaque doublement, il est multiplié par $2^m$ — où $m$ est la petite taille du couple — et par un terme de l'ordre de $m$, si bien que ses trois facteurs ne se ressemblent pas entre eux. Aucune observation de taille finie ne transforme cette différence de nature en théorème — c'est justement ce que le §7 va préciser.

## 7. La borne annoncée, et ce qu'elle ne contredit pas à petite taille

L'article annonce une borne inférieure **superpolynomiale** pour la permanente : tout circuit qui la calcule a une taille qui dépasse, à partir d'un certain rang $m_0$, une fonction de la forme

$$B_c(m) = m^{\,c \,(\log_2 m)^{1/3}} \;=\; 2^{\,c\,(\log_2 m)^{4/3}}$$

pour une constante $c > 0$ dont l'énoncé n'annonce **aucune valeur**, pas plus qu'il n'en annonce pour $m_0$ : l'article écrit « this is why Theorem 1.1 comes with no value for $m_0$ » (p. 8) et « The constant $m_0$ in our theorems is not effective » (p. 60). C'est la forme de la borne que le formulaire Lean encode (`Beyond`), et c'est la seule chose que nous puissions confronter à un calcul : **le circuit de Ryser contredit-il la borne, à une taille où l'on peut encore le construire ?**

La réponse doit être lue dans le bon sens, et c'est le point que cette section veut rendre maniable. À petite taille, un circuit exponentiel est *petit* : $2^m$ portes pour $m = 6$, c'est moins de mille. Rien n'interdit donc à $B_c$ d'être **plus grande** que lui. Tant que la borne dépasse le circuit de Ryser, le théorème est **muet** sur ces tailles : c'est sa zone de silence, en dessous de $m_0$. Le théorème n'affirme rien avant $m_0$ ; ce qu'il affirme ensuite, c'est précisément que la borne dépasse tout circuit.

Ce qui le **falsifierait** est donc le contraire : qu'un circuit connu — celui de Ryser — reste **au-dessus** de $B_c$ pour des $m$ arbitrairement grands. Autrement dit, $m_0$ doit être strictement supérieur au dernier rang où le circuit de Ryser dépasse encore la borne. La cellule mesure ce dernier rang pour quelques valeurs de $c$, puis regarde s'il arrive un rang, dans la fenêtre balayée, où le circuit repasse au-dessus. Elle n'en trouve aucun : c'est cohérent avec l'énoncé, et cela ne le prouve pas.

In [8]:
def log2_taille_ryser(m: int) -> float:
    return math.log2(taille_ryser(m))


def log2_borne(c: float, m: int) -> float:
    """log2 de B_c(m) = m^(c (log2 m)^(1/3)) = 2^(c (log2 m)^(4/3))."""
    return c * math.log2(m) ** (4 / 3)


print("m      log2 du circuit de Ryser   log2 B_1   log2 B_10")
for m in (2, 4, 8, 16, 32, 64, 256, 1024, 4096):
    print(f"{m:5d} {log2_taille_ryser(m):24.3f} {log2_borne(1.0, m):10.3f} {log2_borne(10.0, m):10.3f}")
print()
FENETRE = 1 << 12
print(f"balayage de m = 1 a {FENETRE} : plus grand m ou la borne depasse le circuit de Ryser")
for c in (1.0, 10.0, 100.0, 1000.0):
    trouves = [m for m in range(1, FENETRE + 1) if log2_borne(c, m) >= log2_taille_ryser(m)]
    dernier = max(trouves) if trouves else None
    bord = " -- bord de la fenetre, le franchissement est au-dela" if dernier == FENETRE else ""
    print(f"   c = {c:8.1f} : {dernier}{bord}")
print()
print("apres le dernier point contredit pour c = 10, la borne repasse-t-elle au-dessus ?")
print("   (m : nombre de valeurs de m au-dela de 126 ou la borne depasse encore le circuit)")
retours = [m for m in range(127, FENETRE + 1) if log2_borne(10.0, m) >= log2_taille_ryser(m)]
print("   ", len(retours))
print()
print("autour du franchissement pour c = 10 (log2) :")
for m in (124, 125, 126, 127):
    print(f"   m = {m} : borne {log2_borne(10.0, m):9.3f}   circuit de Ryser {log2_taille_ryser(m):9.3f}")

m      log2 du circuit de Ryser   log2 B_1   log2 B_10
    2                    3.807      1.000     10.000
    4                    7.000      2.520     25.198
    8                   12.039      4.327     43.267
   16                   21.022      6.350     63.496
   32                   38.011      8.550     85.499
   64                   71.006     10.903    109.027
  256                  265.001     16.000    160.000
 1024                 1035.000     21.544    215.443
 4096                 4109.000     27.473    274.731

balayage de m = 1 a 4096 : plus grand m ou la borne depasse le circuit de Ryser
   c =      1.0 : 1
   c =     10.0 : 125
   c =    100.0 : 2523
   c =   1000.0 : 4096 -- bord de la fenetre, le franchissement est au-dela

apres le dernier point contredit pour c = 10, la borne repasse-t-elle au-dessus ?
   (m : nombre de valeurs de m au-dela de 126 ou la borne depasse encore le circuit)
    0

autour du franchissement pour c = 10 (log2) :
   m = 124 : borne   132.

### Lecture du résultat : pour $c = 10$, la borne est au-dessus jusqu'à $m = 125$, puis en dessous

Le tableau se lit en base 2 : $B_{10}$ à $m = 64$ vaut encore $2^{109.027}$, au-dessus des $2^{71.006}$ portes du circuit de Ryser ; à $m = 256$, la situation est renversée — $2^{160.000}$ contre $2^{265.001}$. Entre les deux, la borne et le circuit se sont croisés. La borne croît comme $(\log_2 m)^{4/3}$, le circuit comme $m$ dans le logarithme : le croisement est une question de rang, pas de constante.

Le balayage le chiffre. Pour $c = 10$, le dernier rang où la borne dépasse le circuit est 125 ; à 126, le circuit repasse devant, et la ligne suivante répond 0 — la borne ne revient au-dessus à aucune autre valeur jusqu'au bord de la fenêtre, à 4096. Autour du franchissement, l'écart est mince : il se referme, puis s'inverse — 133.033 contre 132.969 à $m = 125$, puis 133.326 contre 133.980 à $m = 126$.

Pour $c = 100$, le dernier rang contredit monte à 2523 ; pour $c = 1$, il tombe à 1. C'est ce que dit l'énoncé : la constante n'est pas annoncée, et un $c$ plus grand élargit la zone où la borne reste au-dessus du seul circuit que nous sachions construire. La valeur 4096 pour $c = 1000$ n'est pas une mesure — elle porte la mention de bord et dit que le franchissement n'a pas eu lieu dans la fenêtre.

Reste la lecture, qui est le point de la section : si le théorème est vrai pour une constante $c$, alors son $m_0$ est strictement supérieur au dernier rang où le circuit de Ryser dépasse encore la borne — ici 125. Le calcul ne dit pas que le théorème est vrai ; il dit que le seul circuit que nous sachions écrire ne le contredit pas, dans la seule fenêtre où nous puissions l'écrire.

## 8. Pourquoi le théorème exclut $p = 2$

Sur un corps de caractéristique 2, $+1 = -1$. Le signe qui distingue la permanente du déterminant **disparaît** : la somme signée et la somme non signée ont les mêmes coefficients, donc les deux polynômes sont égaux. La conséquence est immédiate et fatale pour toute séparation : si $\mathrm{perm} = \det$ en caractéristique 2, et si le déterminant a des circuits de taille polynomiale, alors la permanente aussi. Aucune borne inférieure superpolynomiale ne peut exister là.

C'est exactement ce que l'article écrit : « In characteristic two the permanent coincides with the determinant; since the determinant has polynomial-size circuits, the theorem does not apply there. » (théorème 1.1, p. 2). Le formulaire Lean porte la même restriction dans sa **seconde** clause, celle de la permanente : `p ≠ 2 → Computes (perm F m) s → Beyond c m s`. La **première** clause, elle, porte sur le polynôme du cycle hamiltonien, sans aucune condition sur $p$ : `Computes (hc F m) s → Beyond c m s`. La note 1 de l'article (p. 1) en donne la raison : « The permanent is VNP-complete for fields of characteristic other than 2, while the Hamiltonian cycle polynomial is VNP-complete for all fields. »

La cellule vérifie trois choses distinctes, qu'il ne faut pas confondre :

- l'identité **de polynômes** modulo 2, pour $m$ jusqu'à 5 ;
- le même test modulo 3 et 5, qui doit **échouer** (les polynômes y diffèrent) ;
- l'égalité **exhaustive** sur toutes les matrices à coefficients dans $\mathbb{F}_2$ pour $m \le 3$.

In [9]:
print("identite de polynomes modulo 2 (perm contre det) :")
for m in range(1, 6):
    perm2 = reduire_modulo(polynome_perm(m), 2)
    det2 = reduire_modulo(polynome_perm(m, signe=True), 2)
    print(f"   m = {m} : egaux -> {perm2 == det2}  ({len(perm2)} monomes apres reduction)")
print()
print("controle negatif (les polynomes doivent y differer) :")
for p in (3, 5):
    identiques = sum(reduire_modulo(polynome_perm(m), p) == reduire_modulo(polynome_perm(m, signe=True), p)
                     for m in range(2, 6))
    print(f"   modulo {p} : {identiques} valeur(s) de m sur 4 ou les deux polynomes coincident")
print()
f2 = Anneau(2)
print("egalite exhaustive sur F_2 :")
for m in (1, 2, 3):
    egaux = 0
    for plat in product((0, 1), repeat=m * m):
        A = [list(plat[i * m:(i + 1) * m]) for i in range(m)]
        if f2.egal(permanente_par_definition(A, f2), determinant_par_definition(A, f2)):
            egaux += 1
    print(f"   m = {m} : {egaux} matrices sur {1 << (m * m)}")
print()
print("controle negatif exhaustif, matrices 2 x 2 :")
for p in (3, 5):
    anneau = Anneau(p)
    total = p ** 4
    egaux = 0
    for plat in product(range(p), repeat=4):
        A = [list(plat[0:2]), list(plat[2:4])]
        if anneau.egal(permanente_par_definition(A, anneau), determinant_par_definition(A, anneau)):
            egaux += 1
    print(f"   F_{p} : {egaux} matrices sur {total} avec perm == det, {total - egaux} ou ils different")

identite de polynomes modulo 2 (perm contre det) :
   m = 1 : egaux -> True  (1 monomes apres reduction)
   m = 2 : egaux -> True  (2 monomes apres reduction)
   m = 3 : egaux -> True  (6 monomes apres reduction)
   m = 4 : egaux -> True  (24 monomes apres reduction)
   m = 5 : egaux -> True  (120 monomes apres reduction)

controle negatif (les polynomes doivent y differer) :
   modulo 3 : 0 valeur(s) de m sur 4 ou les deux polynomes coincident
   modulo 5 : 0 valeur(s) de m sur 4 ou les deux polynomes coincident

egalite exhaustive sur F_2 :
   m = 1 : 2 matrices sur 2
   m = 2 : 16 matrices sur 16
   m = 3 : 512 matrices sur 512

controle negatif exhaustif, matrices 2 x 2 :
   F_3 : 45 matrices sur 81 avec perm == det, 36 ou ils different
   F_5 : 225 matrices sur 625 avec perm == det, 400 ou ils different


### Lecture du résultat : égaux modulo 2, distincts modulo 3 et modulo 5

Le premier bloc compare deux **polynômes** réduits modulo 2, et non des valeurs : pour $m$ de 1 à 5, la permanente générique et le déterminant générique y sont le même polynôme. Le nombre de monômes restants — 1, 2, 6, 24, 120 — est celui d'avant la réduction : aucun monôme ne disparaît, aucune fusion n'a lieu, et seuls les coefficients $\pm 1$ sont devenus égaux. L'identité porte donc sur les coefficients, pas sur le support.

Le deuxième bloc est le témoin négatif, et sans lui le premier ne prouverait rien : un test incapable de distinguer les deux polynômes répondrait `True` pour la même raison partout. Modulo 3 et modulo 5, pour $m$ de 2 à 5 — les quatre tailles que la cellule soumet à ce contrôle —, le nombre de coïncidences est 0.

Le troisième bloc quitte les polynômes pour les matrices : sur $\mathbb{F}_2$, **toutes** les matrices sont d'accord avec l'identité — 2 sur 2 pour $m = 1$, 16 sur 16 pour $m = 2$, 512 sur 512 pour $m = 3$. L'égalité polynomiale entraîne l'égalité en tout point, et l'énumération exhaustive le confirme sans hypothèse.

Le quatrième bloc en est le pendant : sur les matrices $2 \times 2$ à coefficients dans $\mathbb{F}_3$, 45 matrices sur 81 donnent la même permanente et le même déterminant, et 36 ne les donnent pas ; sur $\mathbb{F}_5$, 225 sur 625, et 400 qui diffèrent. Le compte s'explique : pour $\begin{pmatrix} a & b \\ c & d \end{pmatrix}$, la différence $\mathrm{perm} - \det$ vaut $2bc$, qui s'annule dès que $b$ ou $c$ est nul — d'où les 36 $= 2 \times 2 \times 3 \times 3$ matrices de $\mathbb{F}_3$ qui séparent les deux polynômes, et les 400 $= 4 \times 4 \times 5 \times 5$ de $\mathbb{F}_5$. Une coïncidence sur une matrice n'est pas une identité : c'est l'identité de polynômes, et elle seule, qui rend compte de la caractéristique 2.

## 9. La forme VNP : le certificat qui définit la classe

Pourquoi la permanente est-elle dans VNP ? La classe se définit par une **somme sur un certificat de taille polynomiale** : une somme sur $e \in \{0,1\}^m$, indexée par un vecteur de $m$ bits — exactement la forme qu'utilisent les identités de comptage. En posant

$$g(A, e) = \prod_{i=1}^{m} \Big( \sum_{j=1}^{m} e_j\, a_{ij} \Big) \cdot \prod_{j=1}^{m} \big( 2 e_j - 1 \big)$$

l'identité s'écrit comme une somme nue, sans facteur extérieur :

$$\mathrm{perm}(A) = \sum_{e \in \{0,1\}^m} g(A, e)$$

C'est la forme qu'exige la définition de `VNP` du formulaire Lean (§10 : la famille est une **somme** sur `Fin (r n) → Bool`, sans coefficient devant), et celle que le texte de l'article retient pour la classe (p. 4). L'article écrit aussi, p. 60, la forme **équivalente** où le facteur $(1 - 2e_j)$ reste sous son signe d'origine, avec $(-1)^m$ devant la somme : les deux écritures coïncident parce que $2e_j - 1 = -(1 - 2e_j)$ et qu'il y a $m$ facteurs, donc

$$\prod_{j=1}^{m} (2 e_j - 1) = (-1)^{m - |e|}$$

— le signe de Ryser, déjà rencontré au §5.

La cellule vérifie l'identité sur quatre anneaux, évalue le circuit de $g_m$ contre la fonction (le « témoin » : les entrées $e$ sont des variables du circuit, pas des constantes), et compte ses portes.

**Un piège à ne pas manquer.** Cette expression **ne dégénère pas** en caractéristique 2 : le signe y disparaît, et l'identité y reste vraie, comme la cellule va l'imprimer. Le fait que le théorème exclue $p = 2$ ne vient donc pas d'ici. Il vient de ce que la permanente **coïncide avec le déterminant** en caractéristique 2 (§8) : puisqu'aucune borne superpolynomiale n'y est possible pour elle, la séparation est établie sur le **polynôme du cycle hamiltonien**, et la preuve du corollaire M.4 (p. 62) porte effectivement sur `hc`. Confondre les deux serait chercher dans cette identité une obstruction qui n'y est pas.

In [10]:
def valeur_g(A, e, anneau: Anneau):
    """g(A, e) = prod_i (somme_j e_j a_ij) * prod_j (2 e_j - 1) : le terme du certificat e."""
    m = len(A)
    g = anneau.un()
    for i in range(m):
        t = anneau.zero()
        for j in range(m):
            t = anneau.add(t, anneau.mul(e[j], A[i][j]))
        g = anneau.mul(g, t)
    for j in range(m):
        g = anneau.mul(g, anneau.reduire(2 * e[j] - 1))
    return g


def forme_vnp(A, anneau: Anneau):
    """perm(A) = somme sur e dans {0,1}^m de g(A, e) : une somme nue, sans facteur exterieur."""
    total = anneau.zero()
    for e in product((0, 1), repeat=len(A)):
        total = anneau.add(total, valeur_g(A, e, anneau))
    return total


def circuit_g(m: int) -> Circuit:
    """Circuit de g_m : entrees a_ij (indices i * m + j) puis e_j (indices m^2 + j)."""
    ops = OperationsCircuit()
    c = ops.circuit
    deux = c.const(2)
    x = [[ops.entree(i * m + j) for j in range(m)] for i in range(m)]
    e = [ops.entree(m * m + j) for j in range(m)]
    g = None
    for i in range(m):
        t = c.mul(e[0], x[i][0])
        for j in range(1, m):
            t = c.add(t, c.mul(e[j], x[i][j]))
        g = t if g is None else c.mul(g, t)
    for j in range(m):
        u = c.add(c.mul(deux, e[j]), ops.moins_un)       # 2 e_j - 1
        g = c.mul(g, u)
    return c


def portes_utiles(c: Circuit) -> set:
    """Ensemble des portes dont depend la sortie : la sortie et ses ancetres."""
    utiles = set()
    pile = [len(c.portes) - 1]
    while pile:
        i = pile.pop()
        if i in utiles:
            continue
        utiles.add(i)
        p = c.portes[i]
        if p[0] in ("add", "mul"):
            pile.extend((p[1], p[2]))
    return utiles


gen = GENERATEUR.Random(SEED + 3)
print("forme VNP contre la definition :")
for anneau in (Anneau(0), Anneau(2), Anneau(3), Anneau(5)):
    bons = 0
    for m in range(1, 5):
        for _ in range(10):
            A = matrice_aleatoire(m, anneau, gen, 4)
            if anneau.egal(forme_vnp(A, anneau), permanente_par_definition(A, anneau)):
                bons += 1
    print(f"   {str(anneau):4s} : {bons} egalites sur 40")
print()
print("circuit de g_m evalue contre g(A, e) :")
echecs_g = 0
essais_g = 0
for anneau in (Anneau(0), Anneau(2), Anneau(3), Anneau(5)):
    for m in range(1, 5):
        circuit = circuit_g(m)
        for _ in range(10):
            A = matrice_aleatoire(m, anneau, gen, 4)
            e = [gen.randrange(2) for _ in range(m)]
            plat = [A[i][j] for i in range(m) for j in range(m)] + e
            essais_g += 1
            if not anneau.egal(circuit.evaluer(plat, anneau), valeur_g(A, e, anneau)):
                echecs_g += 1
print(f"   {essais_g} essais, {echecs_g} echec(s)")
print()
print("taille du circuit de g_m (3m^2 + 4m + 1 portes) :")
for m in range(1, 7):
    print(f"   m = {m} : {len(circuit_g(m)):3d} portes, forme close {3 * m * m + 4 * m + 1:3d}")
print()
print("facteur entre les deux facons de compter (note 2 de l'article) :")
for nom, famille in (("Ryser", circuit_ryser), ("determinant", circuit_determinant)):
    pire = 0.0
    for m in range(1, 11):
        circuit = famille(m)
        u = portes_utiles(circuit)
        operations = sum(1 for i in u if circuit.portes[i][0] in ("add", "mul"))
        if operations:
            pire = max(pire, len(u) / operations)
    print(f"   {nom:12s} : portes utiles / (additions + multiplications), au pire {pire:.3f}")

forme VNP contre la definition :
   Z    : 40 egalites sur 40
   F_2  : 40 egalites sur 40
   F_3  : 40 egalites sur 40
   F_5  : 40 egalites sur 40

circuit de g_m evalue contre g(A, e) :
   160 essais, 0 echec(s)

taille du circuit de g_m (3m^2 + 4m + 1 portes) :
   m = 1 :   8 portes, forme close   8
   m = 2 :  21 portes, forme close  21
   m = 3 :  40 portes, forme close  40
   m = 4 :  65 portes, forme close  65
   m = 5 :  96 portes, forme close  96
   m = 6 : 133 portes, forme close 133

facteur entre les deux facons de compter (note 2 de l'article) :
   Ryser        : portes utiles / (additions + multiplications), au pire 1.556
   determinant  : portes utiles / (additions + multiplications), au pire 2.000


### Lecture du résultat : une somme nue, et un témoin de taille quadratique

Le premier bloc vérifie l'identité $\mathrm{perm}(A) = \sum_e g(A, e)$ : 40 égalités sur 40 sur chacun des quatre anneaux. Deux détails y comptent. La somme n'a **aucun facteur devant elle** — c'est la forme qu'exige la définition de `VNP`, qui somme `Fin (r n) → Bool` sans coefficient. Et l'identité survit en caractéristique 2, où $2e_j - 1$ vaut toujours 1 : le facteur de signe s'y écrase, et la somme reste pourtant égale à la permanente. Le théorème exclut $p = 2$ pour une autre raison — la coïncidence avec le déterminant, mesurée au §8.

Le deuxième bloc change le statut des $e_j$ : dans le circuit de $g_m$, ce ne sont plus des constantes qu'on énumère mais des **variables**, et 160 essais confirment que le circuit rend bien $g(A, e)$ quand on lui donne $A$ et $e$. C'est ce qui fait de $g$ un certificat au sens de la classe : on somme sur les valeurs d'un petit circuit, on ne les fixe pas une à une.

Le troisième bloc donne le prix de ce témoin : $3m^2 + 4m + 1$ portes — 8 pour $m = 1$, 133 pour $m = 6$ — un polynôme du second degré, que la cellule trouve égal à la taille construite à chaque taille testée. Le témoin est donc petit et bon marché ; la dépense est dans la somme, qui compte $2^m$ évaluations.

Le dernier bloc mesure le facteur de la note 2 de l'article entre deux façons de compter la taille. En ne retenant, parmi les portes utiles, que les additions et multiplications, la taille du circuit de Ryser tombe d'un facteur au plus 1.556, celle du circuit du déterminant d'un facteur au plus 2.000 — pour $m$ de 1 à 10, l'étendue que se donne la cellule. L'écart vient des variables et des constantes, que ce comptage ne retient pas et qui pèsent d'autant plus que la matrice est petite. L'article annonce « au plus trois » (note 2, p. 4) : la marge est large pour ces deux constructions, et le mesurer vaut mieux que le supposer.

## 10. Les énoncés formels, lus au commit

Le dépôt `anthropics/formal-math` publie, pour son projet `vp-vnp`, un fichier `Statement.lean` qui ne contient **que des définitions et des énoncés**, importe Mathlib seul, et sert de contrat : la preuve vit ailleurs. Nous le lisons au commit `11b6c7e3c927df2658f513b6b68ea7f8929c62c5`, sans compiler quoi que ce soit.

Six points portent l'essentiel du sens, et méritent d'être lus mot à mot :

- `Computes f s` dit qu'il existe une liste de $s$ portes, chacune étant une variable, une constante, une somme ou un produit de portes antérieures, dont **l'une** est $f$ — pas nécessairement la dernière. C'est le modèle du §4, écrit formellement : variables et constantes comptées, aucune soustraction.
- `perm` et `hc` sont les deux polynômes génériques : la permanente, sommée sur toutes les permutations, et le polynôme du cycle hamiltonien, sommé sur celles qui forment un seul cycle. Leurs écritures n'ont pas la même orientation ; c'est le point vérifié au §3.
- `Beyond c m s` est la borne : la taille $s$ doit dépasser $m^{c\,(\log_2 m)^{1/3}}$, et l'inégalité est **stricte**.
- `MainTheorem` est l'énoncé principal : une même constante $c$ convient ; pour chaque caractéristique il existe un $m_0$ au-delà duquel, sur tout corps de cette caractéristique, tout circuit dépasse la borne — avec une clause par polynôme.
- `Approximates` et `MainTheoremApprox` en donnent la forme approchée : un circuit qui calcule $f + \varepsilon\,g$ — un polynôme sur les séries de Laurent en $\varepsilon$ — au lieu de $f$ tombe déjà sous la borne.
- `VP` et `VNP` sont les deux classes : les familles de polynômes à circuits polynomiaux, et celles qui s'écrivent comme une somme, sur un certificat de taille polynomiale, d'une telle famille.

Le texte ci-dessous est recopié **verbatim** du fichier au commit cité : c'est une citation, pas une reformulation.

In [11]:
# --- Statement.lean, lu au commit 11b6c7e3c927df2658f513b6b68ea7f8929c62c5 ---
extraits = {
"Computes (le modele de circuit, formellement)":
"""def Computes {R : Type u} {V : Type} [CommSemiring R] (f : MvPolynomial V R) (s : ℕ) : Prop :=
  ∃ l : ℕ → MvPolynomial V R,
    (∀ k < s, (∃ v : V, l k = X v) ∨ (∃ a : R, l k = C a) ∨
      (∃ i < k, ∃ j < k, l k = l i + l j) ∨ (∃ i < k, ∃ j < k, l k = l i * l j)) ∧
    ∃ k < s, l k = f""",
"perm (la permanente generique)":
"""noncomputable def perm (F : Type u) [CommSemiring F] (m : ℕ) : MvPolynomial (Fin m × Fin m) F :=
  ∑ σ : Equiv.Perm (Fin m), ∏ i, X (σ i, i)""",
"hc (le polynome du cycle hamiltonien)":
"""open Classical in
noncomputable def hc (F : Type u) [CommSemiring F] (m : ℕ) : MvPolynomial (Fin m × Fin m) F :=
  ∑ σ : Equiv.Perm (Fin m) with σ.IsCycleOn Set.univ, ∏ i, X (i, σ i)""",
"Beyond (la borne annoncee)":
"""def Beyond (c : ℝ) (m s : ℕ) : Prop := (m : ℝ) ^ (c * Real.logb 2 m ^ (1 / 3 : ℝ)) < s""",
"MainTheorem (l'enonce du theoreme principal)":
"""def MainTheorem : Prop :=
  ∃ c : ℝ, 0 < c ∧ ∀ p : ℕ, ∃ m₀ : ℕ, ∀ (F : Type u) [Field F] [CharP F p], ∀ m ≥ m₀, ∀ s : ℕ,
    (Computes (hc F m) s → Beyond c m s) ∧ (p ≠ 2 → Computes (perm F m) s → Beyond c m s)""",
"Approximates puis MainTheoremApprox (la forme approchee du theoreme)":
"""def Approximates {F : Type u} {V : Type} [Field F] (f : MvPolynomial V F) (s : ℕ) : Prop :=
  ∃ q : MvPolynomial V (LaurentSeries F), Computes q s ∧
    ∀ (d : V →₀ ℕ) (i : ℤ), i ≤ 0 → (coeff d q).coeff i = if i = 0 then coeff d f else 0

def MainTheoremApprox : Prop :=
  ∃ c : ℝ, 0 < c ∧ ∀ p : ℕ, ∃ m₀ : ℕ, ∀ (F : Type u) [Field F] [CharP F p], ∀ m ≥ m₀, ∀ s : ℕ,
    (Approximates (hc F m) s → Beyond c m s) ∧ (p ≠ 2 → Approximates (perm F m) s → Beyond c m s)""",
"VP (la classe des familles a circuits polynomiaux)":
"""def VP (F : Type u) [CommSemiring F] : Set (Family F) :=
  {f | ∃ a : ℕ, ∀ n, Fintype.card (f.vars n) ≤ n ^ a + a ∧ (f.poly n).totalDegree ≤ n ^ a + a ∧
    Computes (f.poly n) (n ^ a + a)}""",
"VNP (la classe des familles a certificat polynomial)":
"""def VNP (F : Type u) [CommSemiring F] : Set (Family F) :=
  {f | ∃ (r : ℕ → ℕ) (g : ∀ n, MvPolynomial (f.vars n ⊕ Fin (r n)) F),
    ({ vars := fun n => f.vars n ⊕ Fin (r n), poly := g } : Family F) ∈ VP F ∧
      ∀ n, f.poly n = ∑ e : Fin (r n) → Bool, aeval (Sum.elim X fun i => if e i then 1 else 0) (g n)}""",
"Separation / SeparationClosure (les deux enonces de separation)":
"""def Separation : Prop := ∀ (F : Type u) [Field F], VP F ⊂ VNP F

def SeparationClosure : Prop := ∀ (F : Type u) [Field F], ¬ VNP F ⊆ VPBar F""",
}
for nom, code in extraits.items():
    print(f"--- {nom} ---")
    print(code)
    print()
print("Lecture : Beyond est STRICT (<) ; la clause p != 2 porte la permanente,")
print("la premiere clause porte hc (le polynome du cycle hamiltonien, defini au meme fichier).")
print()
print("Les quatre enonces et le resultat numerote de l'article qu'ils reprennent (annexe Q, p. 66) :")
print("   MainTheorem       = Theorem M.3")
print("   MainTheoremApprox = premiere assertion du Theorem N.5")
print("   Separation        = Corollary M.4")
print("   SeparationClosure = seconde assertion du Theorem N.5")

--- Computes (le modele de circuit, formellement) ---
def Computes {R : Type u} {V : Type} [CommSemiring R] (f : MvPolynomial V R) (s : ℕ) : Prop :=
  ∃ l : ℕ → MvPolynomial V R,
    (∀ k < s, (∃ v : V, l k = X v) ∨ (∃ a : R, l k = C a) ∨
      (∃ i < k, ∃ j < k, l k = l i + l j) ∨ (∃ i < k, ∃ j < k, l k = l i * l j)) ∧
    ∃ k < s, l k = f

--- perm (la permanente generique) ---
noncomputable def perm (F : Type u) [CommSemiring F] (m : ℕ) : MvPolynomial (Fin m × Fin m) F :=
  ∑ σ : Equiv.Perm (Fin m), ∏ i, X (σ i, i)

--- hc (le polynome du cycle hamiltonien) ---
open Classical in
noncomputable def hc (F : Type u) [CommSemiring F] (m : ℕ) : MvPolynomial (Fin m × Fin m) F :=
  ∑ σ : Equiv.Perm (Fin m) with σ.IsCycleOn Set.univ, ∏ i, X (i, σ i)

--- Beyond (la borne annoncee) ---
def Beyond (c : ℝ) (m s : ℕ) : Prop := (m : ℝ) ^ (c * Real.logb 2 m ^ (1 / 3 : ℝ)) < s

--- MainTheorem (l'enonce du theoreme principal) ---
def MainTheorem : Prop :=
  ∃ c : ℝ, 0 < c ∧ ∀ p : ℕ, ∃ m₀ : ℕ, ∀ (F 

### Lecture du résultat : ce que les énoncés laissent ouvert

La sortie ne calcule rien : elle recopie neuf extraits et une table de correspondance. Ce qu'elle permet de lire, en revanche, est de l'ordre de la logique — quatre points se lisent sur la seule écriture.

`Beyond` est **strict** : `... < s`, pas `≤`. Un circuit exactement de la taille de la borne n'est donc pas contredit par l'énoncé, et un comptage qui tomberait juste sur $B_c(m)$ ne dirait rien.

`MainTheorem` place `∃ c` **avant** `∀ p` et `∃ m₀` **après** : une seule constante pour toutes les caractéristiques, mais un seuil par caractéristique. La clause `p ≠ 2 →` est attachée au second membre de la conjonction seulement ; la première, celle qui porte `hc`, ne porte aucune condition sur $p$ — c'est la restriction lue au §8, et l'asymétrie est dans l'énoncé, pas dans un commentaire.

`VNP` somme `Fin (r n) → Bool` sans coefficient devant la somme, et `Computes` se contente de `∃ k < s, l k = f` : la porte de sortie peut être n'importe quelle porte de la liste, pas la dernière. Deux conventions plus permissives que le modèle du carnet, et aucune ne change la taille minimale — une liste se tronque après la porte de sortie.

`Approximates` demande enfin un polynôme à coefficients dans les séries de Laurent, `MvPolynomial V (LaurentSeries F)`, et impose `i ≤ 0 → ... = if i = 0 then coeff d f else 0` : les puissances strictement négatives de $\varepsilon$ sont interdites, la puissance nulle doit valoir exactement $f$. Autrement dit, un circuit qui calcule $f + \varepsilon\,g$ tombe déjà sous la borne.

Les quatre dernières lignes sont une table de correspondance, rapportée de l'annexe Q de l'article : `MainTheorem` y répond au Theorem M.3, `MainTheoremApprox` à la première assertion du Theorem N.5, `Separation` au Corollary M.4, `SeparationClosure` à la seconde. Rien dans la sortie n'établit cette correspondance — d'où la ligne « Annexe Q » du tableau de la section suivante.

## 11. Annonce, formalisation, vérification indépendante

Le dépôt publie une chaîne de confiance qu'il faut distinguer pièce par pièce. Voici ce que la lecture des fichiers du commit a montré :

| Pièce | Ce qu'elle contient | Ce qu'elle ne prouve pas |
|---|---|---|
| `Statement.lean` | Les définitions et quatre énoncés (`MainTheorem`, `MainTheoremApprox`, `Separation`, `SeparationClosure`), n'important que Mathlib. | Rien en soi : un énoncé sans preuve est une cible. |
| `Solution.lean` | La preuve, en une cinquantaine de lignes : cinq théorèmes, chacun dérivé d'un résultat du dossier `Paper` — `mainTheorem` de `Paper.thm_allfields`, `separation` de `Paper.cor_allclasses`, `mainTheoremApprox` et `separationClosure` de la première et de la seconde assertion de `Paper.thm_approx`, `permanent_over_real` de `Paper.thm_main`. | Que les résultats de `Paper` soient vrais, ni que les énoncés disent ce qu'on croit. |
| `Challenge.lean` | Les cinq théorèmes de `Solution`, reposés avec `sorry`, pour que l'outil Comparator les compare à la preuve ; d'après l'article, ce sont les seuls `sorry` du développement. | Rien : c'est un formulaire de contrôle. |
| `comparator.json` | Les axiomes permis : `propext`, `Quot.sound`, `Classical.choice` ; `enable_nanoda: true`. | Rien sur la signification des énoncés. |
| Annexe Q de l'article (p. 66 à 70) | La correspondance entre énoncés Lean et résultats numérotés ; le détail des différences d'écriture ; la déclaration de vérification. | La vérification humaine, qui n'a pas eu lieu. |

Quatre passages de l'article portent la limite et la portée du résultat, et nous les citons plutôt que de les paraphraser :

- « The proofs in this paper were found and written by AI. No human mathematician has checked the proofs. » (p. 11)
- « This is the algebraic analogue of the conjecture P ≠ NP, but it does not settle that conjecture. Algebraic circuits must work for all inputs in the field and its extensions – not just 0 and 1 – thus our superpolynomial bound does not apply to Boolean circuits. » (p. 1)
- « The result was discovered by an internal research Claude model and has been formalized in Lean. » (p. 1)
- « Whether the statements say what is meant was judged only by AI readers. No human mathematician has checked the development, the statement file or the proofs. » (p. 66)

Ce que l'article déclare, en revanche, sur le versant machine : Lean 4.33.1 et Mathlib au commit `0df444a360e…` ; aucun `sorry` hors `Challenge.lean` ; aucun axiome au-delà des trois permis ; et, pour le contrôle (Q.1, p. 66) : « Comparator accepted them against Challenge.lean, rechecking the proofs with Lean's kernel and with the independently written checker nanoda. » C'est une mesure sur la **forme**, pas sur le **sens** — et c'est exactement la distinction que la série répète depuis [03d](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb).

Un détail de lecture qui compte pour un lecteur de Mathlib : l'annexe Q recense aussi les écarts d'écriture entre l'article et le formulaire — les nombres $C$, $a$ et $\alpha$ y sont des entiers naturels ; circuits et classes y sont définis sur un semi-anneau commutatif ; la permanente y est écrite $\sum_\sigma \prod_i x_{\sigma(i),i}$ là où la section 1 écrit $x_{i,\sigma(i)}$ (les deux sommes coïncident en remplaçant $\sigma$ par son inverse — c'est ce que la cellule du §3 vérifie ; Q.3, p. 68). Elle recense aussi une restriction d'univers : les théorèmes à préfixe `Paper.` sont énoncés pour des corps dans l'univers le plus bas, « The exception is the given field in Theorems M.3 and N.5, Corollary M.4, Definition N.1 and Lemmas M.1 and N.2, which may lie in any universe. … The theorems of Solution hold for fields in every universe. » (Q.3, p. 66) Enfin elle nomme ce qui n'est **pas** formalisé (p. 70), et la liste est plus longue que les deux points qu'on retient d'habitude : « Remark N.6, the example of flattening ranks after Definition A.1, the remarks that follow the proof of Theorem D.2, the fact that $r$ in the proof of Proposition A.5 is also a limit, and the factor three between the two ways of counting gates in Section 1 are not formalized. » Cinq points, dont le facteur trois du §4 et du §9.

## 12. Exercices

Les trois exercices suivants ne sont pas résolus par le carnet : le code s'arrête à un message et rend `None`. Ils demandent chacun de construire quelque chose que le carnet n'a pas construit, et de le **confronter** à ce qu'il a mesuré.

### Exercice 1 — Compter les couplages parfaits

Écrire `couplages_parfaits(A)` qui, pour une matrice à coefficients 0 ou 1, rend la liste de tous les couplages parfaits du graphe biparti correspondant, chaque couplage étant donné comme une liste de couples `(i, j)`. Vérifier ensuite que le **nombre** de couplages égale `permanente_par_definition(A, Anneau(0))` pour quelques matrices 0/1 — c'est l'identité que le §2 illustre sur la matrice `B`, où l'on lit deux couplages parfaits et une permanente égale à 2.

C'est ce pont que la série voisine développe : [ANALYSE-12](../../SymbolicAI/Lean/Analyse/ANALYSE-12-Comptage-Matching-Lean-Python.ipynb) le présente tel que le lac [`assignment_lean`](../../GameTheory/assignment_lean/README.md) le formalise (`card_eq_permanent`), et [Coalitions-04](../../GameTheory/Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb) en fait un algorithme d'affectation.

In [12]:
def couplages_parfaits(A):
    """Liste des couplages parfaits d'une matrice 0/1, chacun comme une liste de couples (i, j).

    Exercice 1 a completer. Indice : une permutation sigma ne compte que si tous ses
    coefficients valent 1.
    """
    # TODO etudiant : parcourir les permutations et retenir celles dont tous les
    # coefficients sont non nuls.
    print("Exercice 1 a completer")
    return None


print("couplages_parfaits([[0, 1, 1], [1, 0, 1], [1, 1, 0]]) ->", couplages_parfaits([[0, 1, 1], [1, 0, 1], [1, 1, 0]]))
print("resultat attendu quand l'exercice est fait : une liste de deux couplages")

Exercice 1 a completer
couplages_parfaits([[0, 1, 1], [1, 0, 1], [1, 1, 0]]) -> None
resultat attendu quand l'exercice est fait : une liste de deux couplages


### Exercice 2 — La permanente par Ryser, sans circuit

Écrire `permanente_ryser(A, anneau)` qui applique **directement** l'identité du §5, sans construire de circuit : sommer sur les sous-ensembles non vides de colonnes, multiplier les sommes de lignes, appliquer le signe. Confronter le résultat à `permanente_par_definition` sur quelques matrices aléatoires, dans $\mathbb{Z}$ puis dans $\mathbb{F}_3$.

Deux pièges de signe guettent : le facteur global $(-1)^m$, et le signe $(-1)^{|S|}$. Les oublier ne se voit pas sur les petites matrices symétriques — se tester d'abord sur une matrice triangulaire, dont on connaît la permanente (le produit de la diagonale).

In [13]:
def permanente_ryser(A, anneau: Anneau):
    """Permanente par la formule de Ryser, sans circuit.

    Exercice 2 a completer.
    """
    # TODO etudiant : boucler sur les masques non vides, sommer les lignes sur les
    # colonnes du masque, multiplier entre lignes, appliquer le signe.
    print("Exercice 2 a completer")
    return None


A = [[1, 2], [0, 3]]
print("permanente_ryser([[1, 2], [0, 3]], Anneau(0)) ->", permanente_ryser(A, Anneau(0)))
print("valeur attendue quand l'exercice est fait : la meme que la definition, ici le produit de la diagonale")

Exercice 2 a completer
permanente_ryser([[1, 2], [0, 3]], Anneau(0)) -> None
valeur attendue quand l'exercice est fait : la meme que la definition, ici le produit de la diagonale


### Exercice 3 — Évaluer un polynôme de la série

Écrire `evalue_polynome(poly, valeurs)` qui prend un polynôme au format du §3 — un dictionnaire `monome -> coefficient`, un monôme étant un `frozenset` de numéros $u \cdot m + v$ — et un dictionnaire `(u, v) -> valeur`, et rend la valeur du polynôme.

Puis l'appliquer à $\mathrm{hc}_4$ sur deux graphes concrets : le graphe complet $K_4$ et le cycle $C_4$. Le nombre de cycles hamiltoniens de $K_4$ est connu (et le §3 le donne par le nombre de monômes), celui de $C_4$ aussi : c'est le moment de vérifier que l'évaluation retrouve bien les deux, dans les deux orientations de monôme.

In [14]:
def evalue_polynome(poly, valeurs):
    """Valeur d'un polynome {monome: coefficient} sur les valeurs {(u, v): valeur}.

    Exercice 3 a completer.
    """
    # TODO etudiant : pour chaque monome, multiplier les valeurs de ses variables
    # (en retrouvant (u, v) depuis u * m + v), puis sommer avec le coefficient.
    print("Exercice 3 a completer")
    return None


K4 = {(u, v): 1 for u in range(4) for v in range(4) if u != v}
C4 = {(u, v): 1 for u in range(4) for v in ((u + 1) % 4, (u - 1) % 4)}
print("evalue_polynome(polynome_hc(4), K4) ->", evalue_polynome(polynome_hc(4), K4))
print("evalue_polynome(polynome_hc(4), C4) ->", evalue_polynome(polynome_hc(4), C4))

Exercice 3 a completer
evalue_polynome(polynome_hc(4), K4) -> None
Exercice 3 a completer
evalue_polynome(polynome_hc(4), C4) -> None


## 13. Conclusion

| Ce que le carnet établit | Niveau | Où |
|---|---|---|
| Une construction de circuit pour la permanente (Ryser), évaluée contre la définition sur quatre anneaux — 200 essais, 0 échec — et une forme close de sa taille, réduite à une ligne, égale à la taille construite pour $m = 1$ à $12$. | **Mesuré (calcul)** | §5 |
| Une construction de circuit pour le déterminant, sans division, évaluée contre la définition — 240 essais, 0 échec — et de taille polynomiale : une forme close de degré 4, égale à la taille construite pour $m = 1$ à $12$. | **Mesuré (calcul)** | §6 |
| Le contraste de taille : le circuit de Ryser est plus petit que celui du déterminant à $m = 1, 2, 3$ (1 contre 5, 14 contre 15, 45 contre 47) ; il passe devant à $m = 4$ (128 portes contre 121) et vaut environ onze fois la taille du déterminant à $m = 12$. | **Mesuré (calcul)** | §5, §6 |
| La borne annoncée $B_c$ dépasse le circuit de Ryser jusqu'à un rang qui croît avec $c$ — 125 pour $c = 10$, 2523 pour $c = 100$ — puis repasse au-dessous ; rien, dans la fenêtre balayée, ne contredit l'énoncé. | **Mesuré (calcul)** | §7 |
| La permanente et le déterminant coïncident **comme polynômes** modulo 2, et pas modulo 3 ni 5 ; l'égalité est exhaustive sur $\mathbb{F}_2$ pour $m \le 3$. | **Mesuré (calcul)** | §8 |
| La forme VNP de la permanente est exacte sur les quatre anneaux testés, caractéristique 2 comprise — 40 égalités sur 40 par anneau — et le circuit de son terme général l'évalue correctement (160 essais, 0 échec). | **Mesuré (calcul)** | §9 |
| Le contenu de `Statement.lean`, de `Solution.lean`, de `Challenge.lean` et de `comparator.json`, et les écarts d'écriture recensés par l'annexe Q. | **Lu (aucun `lake build` lancé)** | §10, §11 |
| La borne inférieure superpolynomiale, la séparation VP ≠ VNP sur tout corps, et le fait que P ≠ NP n'est pas tranché. | **Cité (annonce des auteurs)** | Bandeau, §11, références |
| La correction de la preuve, et le fait que les énoncés formels disent bien ce qu'ils prétendent dire. | **Hors de portée : non vérifié** | §11 |

**Les limites, en trois phrases.** D'abord, « superpolynomial » ne veut pas dire « visible à petite taille » : jusqu'à $m = 125$ pour $c = 10$, la borne annoncée est encore **au-dessus** de notre circuit de Ryser, et le §7 le montre plutôt qu'il ne le cache. Ensuite, la borne porte sur **tous** les circuits, et seulement au-delà d'un rang $m_0$ dont l'énoncé ne donne aucune valeur : nos constructions mesurent la distance entre la borne et un circuit réel, elles ne la mettent pas à l'épreuve. Enfin, le résultat sépare des classes **algébriques** ; la traduction vers les classes booléennes n'est pas faite, et les auteurs le disent (p. 1).


## Références

**Article** — *VP ≠ VNP over every field*, préprint publié avec sa formalisation Lean (bloc-titre : « CLAUDE ANTHROPIC », p. 1). Le texte du PDF ne porte pas de date de publication propre ; la date retenue ici est celle du commit qui l'archive, `11b6c7e3c927df2658f513b6b68ea7f8929c62c5` (fusion de la pull request #51 du dépôt, 23 novembre 2026, 21 h 43 UTC). Copie archivée au gisement partagé :

`G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\2026 - Anthropic - VP neq VNP over every field (anthropics-formal-math vp-vnp 11b6c7e, preprint).pdf`

(empreinte SHA-256 : `882f5eefb1e067cc8c99ddfe660c10d1d67f9fcee7b69364bd0bab06c0c1f97c`, 1 023 554 octets, 70 pages ; la copie de travail d'où les citations ont été lues a la même empreinte). Les passages cités dans ce carnet y sont lisibles à leur pagination : résumé et note 1 (p. 1) ; théorème 1.1, corollaire 1.2 et travaux antérieurs du §1.2 (p. 2) ; modèle de circuit, note 2, définitions de VP et de VNP (p. 4) ; remarque sur $m_0$ (p. 8) ; section 2.5 (p. 10) ; section 3 (p. 11) ; formule de Ryser et « not effective » (p. 60) ; preuve du corollaire M.4 (p. 62) ; remarque N.6 (p. 64) ; annexe Q (p. 66 à 70).

**Formalisation** — dépôt `anthropics/formal-math`, projet `vp-vnp`, licence Apache-2.0 (Copyright 2026 Anthropic, PBC), lu au commit `11b6c7e3c927df2658f513b6b68ea7f8929c62c5` : `Statement.lean`, `Solution.lean`, `Challenge.lean`, `comparator.json`. Aucun fichier n'est copié dans un lac Lean de ce dépôt ; les extraits du §10 sont des citations.

**Travaux antérieurs, tels que l'article les attribue — et que nous n'avons pas relus.** Les références sont recopiées de la bibliographie de l'article (p. 13 à 16), quelques mentions abrégées ; les attributions sont celles de l'article, et nous ne les avons pas vérifiées à la source.

| Clé | Référence, d'après la bibliographie de l'article | Ce que l'article lui attribue |
|---|---|---|
| `[Val79]` | L. G. Valiant, *Completeness classes in algebra*, STOC 1979, 249–261 (p. 16). | La complétude de la permanente pour VNP sur tout corps où $2 \neq 0$ : « this is a theorem of Valiant [Val79] » (p. 4). |
| `[Val82]` | L. G. Valiant, *Reducibility by algebraic projections*, in *Logic and Algorithmic*, Monogr. Enseign. Math. 30 (1982) (p. 16). | La conjecture, avec `[Val79]` (p. 1) ; la définition des classes VP et VNP, avec `[Val79]` également (p. 4). |
| `[Val79b]` | L. G. Valiant, *The complexity of computing the permanent*, Theoret. Comput. Sci. 8 (1979), 189–201 (p. 16). | L'évaluation de la permanente d'une matrice à coefficients 0 et 1 est #P-complète — l'article précise : « but that is a statement about Boolean computation » (p. 4). |
| `[Rys63]` | H. J. Ryser, *Combinatorial mathematics*, Carus Mathematical Monographs 14, Mathematical Association of America, 1963 (p. 15). | La formule de Ryser (p. 4, p. 60) ; l'une des meilleures bornes **supérieures** connues (p. 1). |
| `[Gly10]` | D. G. Glynn, *The permanent of a square matrix*, European J. Combin. 31 (2010), 1887–1891 (p. 14). | Cité en regard de la formule de Ryser (p. 60) ; l'une des meilleures bornes **supérieures** connues (p. 1). |
| `[Cur26]` | R. Curticapean, *Get permanent results faster by counting stars*, arXiv:2611.03636, 2026 (p. 13). | L'une des meilleures bornes **supérieures** connues (p. 1). |
| `[Ber84]` | S. J. Berkowitz, *On computing the determinant in small parallel time using a small number of processors*, Inform. Process. Lett. 18 (1984), 147–150 (p. 13). | Le déterminant « polynomial en $m$ » (p. 2, p. 10). |
| `[Str73]` | V. Strassen, *Die Berechnungskomplexität von elementarsymmetrischen Funktionen und von Interpolationskoeffizienten*, Numer. Math. 20 (1973), 238–251 (p. 16). | La meilleure borne **inférieure** antérieure pour un polynôme explicite, $\Omega(n \log d)$ (p. 2). |
| `[BS83]` | W. Baur et V. Strassen, *The complexity of partial derivatives*, Theoret. Comput. Sci. 22 (1983), 317–330 (p. 13). | La même borne **inférieure** (p. 2). |

*Carnet écrit le 25 novembre 2026 ; les faits datés qu'il cite — commit, pagination, contenu des fichiers — sont ceux de cette date.*

## À retenir

- **Deux sommes, un signe.** La permanente et le déterminant somment les mêmes $m!$ permutations ; seul le signe les sépare, et tout l'écart de difficulté algorithmique vient de là.
- **Un circuit arithmétique se compte.** Variables et constantes comprises, sans soustraction : une négation est un produit par $-1$. Ce sont ces conventions, et pas d'autres, que les comptes du carnet mesurent. L'article en indique une autre, qui ne change une taille que d'un facteur au plus trois (note 2, p. 4) ; sur nos deux constructions, le facteur mesuré au §9 est de 1,556 au plus pour Ryser et de 2,000 au plus pour le déterminant.
- **Ryser donne un circuit réel, exponentiel.** $2^m$ sous-ensembles, des sommes partagées entre sous-ensembles emboîtés ; la taille suit une forme close, $(4m + 1)\,2^{m-1}$ à un terme près, que la cellule du §5 valide jusqu'à $m = 12$. Elle croît bien plus vite que celle du déterminant : de $m = 6$ à $m = 12$, elle est multipliée par 127 quand celle du déterminant l'est par 16.
- **Le déterminant, lui, a des circuits polynomiaux**, et l'on peut en écrire un sans aucune division : la récursion sur les coefficients du polynôme caractéristique, branchée sur un anneau ou sur un constructeur de portes. La taille de celui-ci est un polynôme de degré 4 en $m$.
- **Une borne inférieure ne se lit pas à petite taille.** La borne $B_c$ annoncée est **au-dessus** du circuit de Ryser sur toute une zone initiale — jusqu'à $m = 125$ pour $c = 10$. Le théorème ne parle qu'au-delà d'un rang $m_0$, dont l'énoncé ne donne aucune valeur (p. 8, p. 60). Ce qui le rendrait faux, c'est l'inverse : un circuit connu qui resterait **au-dessus** de la borne pour des $m$ arbitrairement grands.
- **En caractéristique 2, le signe disparaît.** La permanente y coïncide avec le déterminant comme polynôme — mesuré — donc aucune borne superpolynomiale n'est possible pour elle, et le théorème l'exclut lui-même (p. 2). Dans l'énoncé Lean, la restriction $p \neq 2$ porte la **seconde** clause, celle de la permanente ; la **première**, portée par le polynôme du cycle hamiltonien, vaut sur tout corps (note 1, p. 1).
- **Annonce n'est pas vérification.** Le formulaire Lean, la preuve courte qui en dérive, le formulaire Comparator et l'annexe Q forment une chaîne ; le contrôle humain des énoncés et de la preuve n'a pas eu lieu, et l'article le dit lui-même (p. 11, p. 66).

## Vérifiez votre compréhension

<details><summary>1. Pourquoi la taille du circuit de Ryser croît-elle comme $2^m$ et non comme $m!$ ?</summary>

Parce que l'identité de Ryser remplace la somme sur les permutations par une somme sur les **sous-ensembles** de colonnes : il y en a $2^m - 1$ non vides, et chaque terme coûte $m$ sommes de lignes puis $m-1$ multiplications. Les sommes de lignes sont en outre partagées entre sous-ensembles emboîtés, ce qui ramène leur coût à une addition par couple (ligne, sous-ensemble) au lieu de $m$ additions.

</details>

<details><summary>2. Le carnet mesure que le circuit de Ryser est plus petit que la borne $B_c$ jusqu'à un certain rang (125 pour $c = 10$). Cela contredit-il le théorème ?</summary>

Non. Le théorème ne parle qu'au-delà d'un rang $m_0$, dont l'énoncé ne donne aucune valeur (p. 8, p. 60). Tant que la borne est **au-dessus** du circuit de Ryser, il n'a encore rien affirmé. Le théorème serait faux dans le cas contraire : si un circuit connu restait **au-dessus** de la borne pour des $m$ arbitrairement grands, aucun $m_0$ ne conviendrait. Pour $c = 10$, cela impose donc $m_0 > 125$ ; la cellule cherche, après ce rang, un retour de la borne au-dessus du circuit, et n'en trouve pas dans la fenêtre balayée. Rien ne contredit l'énoncé, et rien ne le prouve.

</details>

<details><summary>3. Pourquoi l'égalité permanente = déterminant en caractéristique 2 interdit-elle une borne superpolynomiale pour la permanente sur ces corps ?</summary>

Parce que le déterminant a des circuits de taille polynomiale. Là où les deux polynômes sont égaux, un circuit pour le déterminant **est** un circuit pour la permanente : aucune borne superpolynomiale n'est alors possible pour elle, et l'article le dit dans l'énoncé même du théorème 1.1 (p. 2). L'identité de polynômes modulo 2, mesurée au §8 pour $m \le 5$, vaut sur tout corps de caractéristique 2. Dans l'énoncé Lean, la restriction porte sur la **seconde** clause, celle de la permanente (`p ≠ 2 → …`) ; la **première**, portée par le polynôme du cycle hamiltonien, vaut sur tout corps — il est VNP-complet sur tout corps, caractéristique 2 comprise (note 1, p. 1).

</details>

<details><summary>4. La forme VNP de la permanente dégénère-t-elle en caractéristique 2 ?</summary>

Non — c'est le piège du §9. Dans le terme du certificat, $g(A, e) = \prod_i \big(\sum_j e_j\, a_{ij}\big) \cdot \prod_j (2e_j - 1)$, le second produit vaut $\pm 1$ et n'est autre que le signe de Ryser ; modulo 2, $+1 = -1$, l'identité reste vraie, et la cellule le mesure sur $\mathbb{F}_2$ comme sur les trois autres anneaux. Ce qui interdit la borne en caractéristique 2, c'est tout autre chose : la permanente y est égale au déterminant (p. 2), et la séparation y est établie sur le polynôme du cycle hamiltonien (preuve du corollaire M.4, p. 62).

</details>

<details><summary>5. Le carnet vérifie-t-il la preuve annoncée ?</summary>

Non, à aucun moment. Il lit `Statement.lean` au commit cité et cite l'article ; il ne compile pas, ne rejoue pas la preuve, et ne juge pas si les énoncés formels disent ce qu'ils prétendent. L'annexe Q rapporte que les propositions ont été acceptées par le noyau de Lean et par `nanoda` — « Comparator accepted them against Challenge.lean, rechecking the proofs with Lean's kernel and with the independently written checker nanoda. » (p. 66) — et ajoute qu'aucun mathématicien humain n'a vérifié le développement. C'est une **citation** de ce que les auteurs rapportent, au même titre que le reste.

</details>

## Pour aller plus loin

- [05 — Compter est plus dur que vérifier](Complexity-05-CountingHarder-Permanent-Python.ipynb) et sa [version C#](Complexity-05-CountingHarder-Permanent-CSharp.ipynb) : la permanente comme comptage #P-complet — le palier dont celui-ci part.
- [05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) et sa [version C#](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-CSharp.ipynb) : la même permanente au cœur d'un protocole quantique, le `BosonSampling` d'Aaronson et Arkhipov (2011).
- [ANALYSE-12 — Compter les couplages parfaits](../../SymbolicAI/Lean/Analyse/ANALYSE-12-Comptage-Matching-Lean-Python.ipynb) : le pont que l'exercice 1 fait calculer, relu de près — le théorème `card_eq_permanent` du lac [`assignment_lean`](../../GameTheory/assignment_lean/README.md), ses signatures et ses axiomes.
- [Coalitions-04 — L'algorithme de Kuhn-Munkres](../../GameTheory/Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb) et [Coalitions-05 — Le lake `assignment_lean` par son certificat](../../GameTheory/Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) : sur le même ensemble de permutations, on cherche celle de coût minimal au lieu de les compter. C'est un problème d'optimisation, pas un comptage : sa dualité ne dit rien du nombre de couplages. Coalitions-05 précise elle-même que le module `Counting` du lac — le pont permanent ↔ cardinal — est hors de sa chaîne.
- [Sudoku-15 — Résolution probabiliste](../../Sudoku/Sudoku-15-Infer-Python.ipynb) et sa [version C#](../../Sudoku/Sudoku-15-Infer-CSharp.ipynb) : la formule de Ryser du §5, appliquée à des mineurs 8×8. Un facteur « toutes différentes » d'arité 9 y calcule ses messages exacts par une permanente : elle **somme** les affectations valides, là où la méthode hongroise en **maximise** une.
- [App-21 — Voice leading minimal par affectation](../../Search/Part3-Cas-pratiques/App-21-VoiceLeading-Python.ipynb) et sa [version C#](../../Search/Part3-Cas-pratiques/App-21-VoiceLeading-CSharp.ipynb) : la même affectation de coût minimal, entre deux accords — les coefficients sont des coûts musicaux, et l'on n'y évalue aucune permanente.
- [03d — La réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) : le même exercice de lecture — une annonce, des formalisations, et pour chaque phrase son niveau — appliqué à un autre résultat.